# 12.3 怎麼看聲音隨時間變化？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像用放大鏡逐段看聲音：時間框可重疊**

框長決定分析範圍；hop 決定多久移動一次。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/frames.svg")))

**先想一想：**框400點、hop160點，是不重疊嗎？

聲音頻率會隨時間改變，所以切成短框。框可重疊；hop決定時間分辨率，frame長度決定頻率分析範圍。

**把直覺寫成數學：**frames=N.unfold(size,hop)；相鄰框可共享樣本。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import tone

wave = tone()
frames = wave.unfold(0, 400, 160)
print(frames.shape, "框秒", 400 / 16000, "步秒", 160 / 16000)
assert torch.equal(frames[0, 160:], frames[1, :240])

**如何讀結果：**PyTorch STFT center=True會在兩邊padding，框數可比直接unfold多；要記錄此設置。

**只改一件事：**只把hop改成400。
